Language: **English (default)** | [Polski](lab.ipynb)

# Lab 02 -- classification on concentric circles with an MLP

Companion material to [`README.md`](README.md). Different data than the
lecture (circles, not half-moons), your own `nn.Module` implementation, and
an architecture/activation comparison.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(3)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Data: two concentric circles (given)

The outer ring (radius 1) is class 0, the inner circle (radius 0.5) is
class 1, both with Gaussian noise.

In [ ]:
def make_circles(n_samples, noise, factor, seed):
    g = torch.Generator().manual_seed(seed)
    n0 = n_samples // 2
    n1 = n_samples - n0

    theta0 = 2 * torch.pi * torch.rand(n0, generator=g)
    outer = torch.stack([torch.cos(theta0), torch.sin(theta0)], dim=1)

    theta1 = 2 * torch.pi * torch.rand(n1, generator=g)
    inner = factor * torch.stack([torch.cos(theta1), torch.sin(theta1)], dim=1)

    X = torch.cat([outer, inner], dim=0)
    y = torch.cat([torch.zeros(n0), torch.ones(n1)])
    X = X + noise * torch.randn(X.shape[0], 2, generator=g)

    perm = torch.randperm(X.shape[0], generator=g)
    return X[perm], y[perm]


X, y = make_circles(n_samples=320, noise=0.12, factor=0.5, seed=0)

n = X.shape[0]
perm = torch.randperm(n)
n_test = int(0.2 * n)
test_idx, train_idx = perm[:n_test], perm[n_test:]

X_train, y_train = X[train_idx].to(device), y[train_idx].to(device)
X_test, y_test = X[test_idx].to(device), y[test_idx].to(device)

print(f"N_train={len(train_idx)}, N_test={len(test_idx)}")

colors = ["tab:blue" if v == 0 else "tab:orange" for v in y]
plt.figure(figsize=(4.5, 4.5))
plt.scatter(X[:, 0], X[:, 1], c=colors, s=15, alpha=0.7)
plt.title("Two concentric circles")
plt.xlabel("x1")
plt.ylabel("x2")
plt.gca().set_aspect("equal")
plt.tight_layout()
plt.show()

## 2. Metrics -- your task

Class 1 (inner circle) is the positive class.

In [ ]:
def confusion_counts(preds, y_true):
    """Returns (tp, fp, tn, fn) for 0/1 predictions and 0/1 labels."""
    # TODO: compute TP, FP, TN, FN with tensor comparisons (==, &) and .sum().
    raise NotImplementedError("TODO: confusion_counts")


def metrics_from_preds(preds, y_true):
    """Returns dict {"accuracy", "precision", "recall"}."""
    # TODO: use confusion_counts and the formulas:
    # accuracy = (tp+tn)/(tp+fp+tn+fn), precision = tp/(tp+fp), recall = tp/(tp+fn)
    # (watch out for division by zero when the denominator is 0)
    raise NotImplementedError("TODO: metrics_from_preds")


def predict(model, X):
    """Returns 0/1 predictions (threshold 0.5 on the model's sigmoid logits)."""
    # TODO
    raise NotImplementedError("TODO: predict")


def evaluate(model, X, y_true):
    """Returns metrics_from_preds(predict(model, X), y_true)."""
    # TODO
    raise NotImplementedError("TODO: evaluate")

## 3. Baseline (given)

A single linear layer -- the same kind of model as in the lecture, on
different data.

In [ ]:
torch.manual_seed(0)
baseline_model = nn.Linear(2, 1).to(device)
baseline_optimizer = torch.optim.Adam(baseline_model.parameters(), lr=0.05)
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(400):
    baseline_optimizer.zero_grad()
    logits = baseline_model(X_train).squeeze(-1)
    loss = loss_fn(logits, y_train)
    loss.backward()
    baseline_optimizer.step()

baseline_metrics = evaluate(baseline_model, X_test, y_test)
print(f"Baseline (linear) on the test set: {baseline_metrics}")

## 4. Your MLP -- your task

Implement an `MLP(nn.Module)` class with one hidden layer of configurable
size and configurable activation.

In [ ]:
class MLP(nn.Module):
    """MLP: Linear(2, hidden_size) -> activation -> Linear(hidden_size, 1)."""

    def __init__(self, hidden_size, activation=nn.ReLU):
        super().__init__()
        # TODO: define the layers as self attributes (e.g. self.layer1 = ...).
        # `activation` is a class (e.g. nn.ReLU) - create an instance: activation().
        raise NotImplementedError("TODO: MLP.__init__")

    def forward(self, x):
        # TODO: pass x through layer 1, the activation, layer 2. Return logits
        # (no sigmoid - BCEWithLogitsLoss computes it itself, see the lecture).
        raise NotImplementedError("TODO: MLP.forward")

## 5. Training function -- your task

Generalize the lecture's training loop so it works on any `nn.Module`
passed to it.

In [ ]:
def train_model(model, X_train, y_train, epochs=400, lr=0.05, seed=0):
    """Trains `model` (any nn.Module) with BCEWithLogitsLoss + Adam.

    Returns (model, loss_history).
    """
    # TODO: forward -> loss (BCEWithLogitsLoss) -> backward -> optimizer.step()
    # in a loop over `epochs`, collecting each epoch's loss into a list. See
    # the analogous function in lectures/02_mlp/examples.ipynb (train_classifier).
    raise NotImplementedError("TODO: train_model")

## 6. Experiment: default MLP vs. baseline (given, uses your code)

In [ ]:
default_mlp, default_loss_hist = train_model(
    MLP(hidden_size=16, activation=nn.ReLU), X_train, y_train
)
default_metrics = evaluate(default_mlp, X_test, y_test)

print(f"Baseline (linear):        {baseline_metrics}")
print(f"MLP (hidden=16, ReLU):    {default_metrics}")

## 7. Architecture/activation comparison -- your task

Choose at least three configurations (different hidden size and/or
different activation), including at least one you suspect will be too
weak. Train each and collect the test metrics.

In [ ]:
# TODO: define a list of >= 3 configurations (e.g. dicts with hidden_size and
# activation), train each with train_model(MLP(...), X_train, y_train), and
# collect test metrics (evaluate) in a list `results`, printing them along the way.
raise NotImplementedError("TODO: configuration comparison")

## 8. Evaluation -- your task

Interpret the table from section 7 (Markdown cell, 3-5 sentences): which
configuration was best, was any clearly too weak, and why -- in terms of
what that architecture can and cannot represent.

_(Your answer here.)_

## 9. Misclassification analysis -- your task

The default MLP's decision boundary from section 6, with the misclassified
test points marked (code given).

In [ ]:
def plot_errors(model, X, y_true):
    x_min, x_max = X[:, 0].min() - 0.3, X[:, 0].max() + 0.3
    y_min, y_max = X[:, 1].min() - 0.3, X[:, 1].max() + 0.3
    xx, yy = torch.meshgrid(
        torch.linspace(x_min, x_max, 200),
        torch.linspace(y_min, y_max, 200),
        indexing="xy",
    )
    grid = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1).to(X.device)
    with torch.no_grad():
        probs = torch.sigmoid(model(grid).squeeze(-1)).cpu().reshape(xx.shape)

    preds = predict(model, X)
    correct = preds == y_true

    plt.figure(figsize=(5.5, 5))
    plt.contourf(xx.cpu(), yy.cpu(), probs, levels=20, cmap="RdBu_r", alpha=0.6, vmin=0, vmax=1)
    plt.scatter(
        X[correct, 0].cpu(), X[correct, 1].cpu(),
        c=["tab:blue" if v == 0 else "tab:orange" for v in y_true[correct].cpu()],
        s=15, edgecolor="black", linewidth=0.3, label="correct",
    )
    plt.scatter(
        X[~correct, 0].cpu(), X[~correct, 1].cpu(),
        c="lime", marker="x", s=80, linewidth=2, label="wrong",
    )
    plt.gca().set_aspect("equal")
    plt.legend()
    plt.title("Misclassifications on the test set (default MLP)")
    plt.tight_layout()
    plt.show()


plot_errors(default_mlp, X_test, y_test)

**Task (3-5 sentences, Markdown cell):** where do the misclassified
points lie relative to the decision boundary? Do they look like points
that are "inherently hard" (close to the true boundary, victims of
noise), or rather like evidence that the model systematically
misunderstands something? Justify your answer by looking at the plot.

_(Your answer here.)_

## 10. Conclusions -- your task

Answer (markdown, justified, not one-liners):

1. Does the result of the experiment in section 7 match what the lecture's
   proof about composing affine transformations teaches? How does what
   this experiment shows differ from what the proof showed?

   _(Your answer here.)_

2. If you had to give one sentence of advice to someone designing an MLP
   architecture for a new classification problem, what would it be --
   based on what you observed in this lab, not generalities?

   _(Your answer here.)_